# 02 — TF-IDF Baseline V2 (LIAR)

**Goal:** improve the LIAR binary TF-IDF baseline and make model selection more systematic.

This notebook does four things:

1. load LIAR train / valid / test
2. apply the fixed binary mapping used in this project
3. compare preprocessing and TF-IDF settings on the validation split
4. evaluate the final selected Version 2 model once on the test split

**Fixed binary mapping in this project:**
- REAL (0): `true`, `mostly-true`, `half-true`
- FAKE (1): `barely-true`, `false`, `pants-fire`

**Version 1 reference result:**
- Accuracy: `0.6196`
- Macro-F1: `0.5935`

This notebook is the user's own working baseline notebook for this project.

## Important note about `min_df` and `max_df`

The supervisor asked whether something was wrong because `max_df` looked numerically smaller than `min_df`.

Here that is **not automatically a problem**:

- `min_df=2` means a term must appear in **at least 2 documents**
- `max_df=0.95` means a term is removed if it appears in **more than 95% of documents**

So these two parameters can use **different scales**:
- `min_df` can be an integer count
- `max_df` can be a proportion

What matters is not whether the raw numbers look bigger or smaller, but whether the settings are sensible and whether they improve validation performance.

## 0) Setup

- This notebook fixes the binary label mapping to the project standard: `REAL=0`, `FAKE=1`
- It uses `valid.tsv` for model selection
- It keeps `test.tsv` for final evaluation only

In [ ]:
# Optional on a fresh environment:
# !pip -q install nltk

import re
import ast
from pathlib import Path
from datetime import datetime

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

try:
    from nltk.stem import PorterStemmer
except ImportError:
    raise ImportError("Please install nltk first, e.g. pip install nltk")

DATA_DIR = Path("data/liar")

RANDOM_SEED = 42
LR_MAX_ITER = 2000
LR_C = 1.0

V1_REFERENCE_ACC = 0.6196
V1_REFERENCE_MACRO_F1 = 0.5935

BIN_MAP = {
    "true": 0,
    "mostly-true": 0,
    "half-true": 0,
    "barely-true": 1,
    "false": 1,
    "pants-fire": 1,
}

ID2LABEL = {
    0: "REAL",
    1: "FAKE",
}

STOPWORDS = set(ENGLISH_STOP_WORDS)
STEMMER = PorterStemmer()

print("Parameters:")
print("  DATA_DIR =", DATA_DIR)
print("  LR_MAX_ITER =", LR_MAX_ITER)
print("  LR_C =", LR_C)
print("  RANDOM_SEED =", RANDOM_SEED)
print("  Fixed label mapping: REAL=0, FAKE=1")

## 1) Load LIAR and inspect class balance

This step loads `train.tsv`, `valid.tsv`, `test.tsv`, applies the fixed binary mapping, and prints counts plus percentages.

In [ ]:
train_path = DATA_DIR / "train.tsv"
valid_path = DATA_DIR / "valid.tsv"
test_path  = DATA_DIR / "test.tsv"

cols = [
    "id", "label", "statement", "subject", "speaker", "speaker_job", "state", "party",
    "barely_true_counts", "false_counts", "half_true_counts", "mostly_true_counts",
    "pants_on_fire_counts", "context"
]

def load_tsv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}\n"
            f"Please upload LIAR TSVs to {DATA_DIR}/ or change DATA_DIR."
        )
    return pd.read_csv(path, sep="\t", header=None, names=cols)

def clean_statement(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def prepare_binary(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["statement"] = clean_statement(out["statement"])
    out = out[out["statement"] != ""].copy()
    out["y"] = out["label"].map(BIN_MAP)
    out = out.dropna(subset=["y"]).copy()
    out["y"] = out["y"].astype(int)
    return out

def show_binary_distribution(df: pd.DataFrame, name: str):
    counts = df["y"].value_counts().sort_index()
    total = len(df)
    print(f"\n{name} binary distribution:")
    for k in [0, 1]:
        count = int(counts.get(k, 0))
        pct = 100 * count / total if total else 0
        print(f"  {ID2LABEL[k]} ({k}): {count} ({pct:.2f}%)")

train = load_tsv(train_path)
valid = load_tsv(valid_path)
test  = load_tsv(test_path)

train_p = prepare_binary(train)
valid_p = prepare_binary(valid)
test_p  = prepare_binary(test)

print("Prepared shapes:")
print("  train_p:", train_p.shape)
print("  valid_p:", valid_p.shape)
print("  test_p :", test_p.shape)

show_binary_distribution(train_p, "Train")
show_binary_distribution(valid_p, "Valid")
show_binary_distribution(test_p, "Test")

## 2) Preprocessing functions

The supervisor asked for stronger preprocessing than simple whitespace stripping.

This notebook compares:
- minimal cleanup
- lowercase
- lowercase + stopword removal
- lowercase + stopword removal + stemming

In [ ]:
def preprocess_text(
    text: str,
    lower: bool = True,
    remove_stopwords: bool = False,
    use_stemming: bool = False
) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()

    if lower or remove_stopwords or use_stemming:
        text = text.lower()

    if not remove_stopwords and not use_stemming:
        return text

    tokens = re.findall(r"[a-zA-Z']+", text)

    if remove_stopwords:
        tokens = [tok for tok in tokens if tok not in STOPWORDS]

    if use_stemming:
        tokens = [STEMMER.stem(tok) for tok in tokens]

    return " ".join(tokens)

def preprocess_series(series: pd.Series, config: dict) -> pd.Series:
    return series.apply(lambda x: preprocess_text(x, **config))

def run_single_experiment(
    train_texts,
    train_labels,
    eval_texts,
    eval_labels,
    ngram_range=(1, 1),
    min_df=1,
    max_df=1.0,
    C=1.0
):
    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df
    )

    X_train = vectorizer.fit_transform(train_texts)
    X_eval = vectorizer.transform(eval_texts)

    model = LogisticRegression(
        max_iter=LR_MAX_ITER,
        C=C,
        random_state=RANDOM_SEED
    )

    model.fit(X_train, train_labels)
    preds = model.predict(X_eval)

    acc = accuracy_score(eval_labels, preds)
    macro_f1 = f1_score(eval_labels, preds, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "n_features": X_train.shape[1],
        "vectorizer": vectorizer,
        "model": model,
        "preds": preds,
    }

## 3) Phase 1 — compare preprocessing on the validation split

Keep TF-IDF settings fixed first, so the preprocessing comparison is easier to interpret.

In [ ]:
PREPROCESS_CONFIGS = {
    "P0_minimal": {
        "lower": False,
        "remove_stopwords": False,
        "use_stemming": False,
    },
    "P1_lower": {
        "lower": True,
        "remove_stopwords": False,
        "use_stemming": False,
    },
    "P2_lower_stopwords": {
        "lower": True,
        "remove_stopwords": True,
        "use_stemming": False,
    },
    "P3_lower_stopwords_stemming": {
        "lower": True,
        "remove_stopwords": True,
        "use_stemming": True,
    },
}

phase1_rows = []

for exp_id, (prep_name, prep_cfg) in enumerate(PREPROCESS_CONFIGS.items(), start=1):
    train_texts = preprocess_series(train_p["statement"], prep_cfg)
    valid_texts = preprocess_series(valid_p["statement"], prep_cfg)

    result = run_single_experiment(
        train_texts=train_texts,
        train_labels=train_p["y"],
        eval_texts=valid_texts,
        eval_labels=valid_p["y"],
        ngram_range=(1, 1),
        min_df=1,
        max_df=1.0,
        C=LR_C,
    )

    phase1_rows.append({
        "exp_id": f"E{exp_id}",
        "phase": "preprocess",
        "preprocess": prep_name,
        "ngram_range": "(1,1)",
        "min_df": 1,
        "max_df": 1.0,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
        "n_features": result["n_features"],
    })

phase1_df = pd.DataFrame(phase1_rows).sort_values(
    ["macro_f1", "accuracy"], ascending=[False, False]
).reset_index(drop=True)

phase1_df

## 4) Phase 2 — compare TF-IDF settings on the validation split

After selecting the best preprocessing setup from Phase 1, compare TF-IDF parameter settings.

Here the notebook checks:
- `ngram_range`
- `min_df`
- `max_df`

This directly addresses the supervisor's question about TF-IDF settings.

In [ ]:
best_preprocess_name = phase1_df.iloc[0]["preprocess"]
best_preprocess_cfg = PREPROCESS_CONFIGS[best_preprocess_name]

print("Best preprocessing from Phase 1:", best_preprocess_name)

train_best = preprocess_series(train_p["statement"], best_preprocess_cfg)
valid_best = preprocess_series(valid_p["statement"], best_preprocess_cfg)

tfidf_settings = [
    {"exp_id": "E5", "ngram_range": (1, 2), "min_df": 1, "max_df": 1.0},
    {"exp_id": "E6", "ngram_range": (1, 2), "min_df": 2, "max_df": 1.0},
    {"exp_id": "E7", "ngram_range": (1, 2), "min_df": 2, "max_df": 0.95},
    {"exp_id": "E8", "ngram_range": (1, 2), "min_df": 5, "max_df": 0.95},
]

phase2_rows = []

for cfg in tfidf_settings:
    result = run_single_experiment(
        train_texts=train_best,
        train_labels=train_p["y"],
        eval_texts=valid_best,
        eval_labels=valid_p["y"],
        ngram_range=cfg["ngram_range"],
        min_df=cfg["min_df"],
        max_df=cfg["max_df"],
        C=LR_C,
    )

    phase2_rows.append({
        "exp_id": cfg["exp_id"],
        "phase": "tfidf",
        "preprocess": best_preprocess_name,
        "ngram_range": str(cfg["ngram_range"]),
        "min_df": cfg["min_df"],
        "max_df": cfg["max_df"],
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
        "n_features": result["n_features"],
    })

phase2_df = pd.DataFrame(phase2_rows).sort_values(
    ["macro_f1", "accuracy"], ascending=[False, False]
).reset_index(drop=True)

phase2_df

## 5) Select the best validation configuration and evaluate on the test split

This step does what the supervisor asked:
- use validation for model selection
- keep test for final reporting only

In [ ]:
valid_results_df = pd.concat([phase1_df, phase2_df], ignore_index=True)
valid_results_df = valid_results_df.sort_values(
    ["macro_f1", "accuracy"], ascending=[False, False]
).reset_index(drop=True)

print("All validation results:")
print(valid_results_df)

best_row = valid_results_df.iloc[0]

print("\nSelected best configuration:")
print(best_row)

selected_preprocess_name = best_row["preprocess"]
selected_preprocess_cfg = PREPROCESS_CONFIGS[selected_preprocess_name]
selected_ngram_range = ast.literal_eval(best_row["ngram_range"])
selected_min_df = int(best_row["min_df"])
selected_max_df = float(best_row["max_df"])

train_valid_texts = pd.concat(
    [
        preprocess_series(train_p["statement"], selected_preprocess_cfg),
        preprocess_series(valid_p["statement"], selected_preprocess_cfg),
    ],
    axis=0
)

train_valid_y = pd.concat([train_p["y"], valid_p["y"]], axis=0)
test_texts = preprocess_series(test_p["statement"], selected_preprocess_cfg)

final_result = run_single_experiment(
    train_texts=train_valid_texts,
    train_labels=train_valid_y,
    eval_texts=test_texts,
    eval_labels=test_p["y"],
    ngram_range=selected_ngram_range,
    min_df=selected_min_df,
    max_df=selected_max_df,
    C=LR_C,
)

final_preds = final_result["preds"]

final_acc = final_result["accuracy"]
final_macro_f1 = final_result["macro_f1"]

print("\nFinal Version 2 test result:")
print(f"Accuracy: {final_acc:.4f}")
print(f"Macro-F1: {final_macro_f1:.4f}")

print("\nDelta vs Version 1:")
print(f"Accuracy delta: {final_acc - V1_REFERENCE_ACC:+.4f}")
print(f"Macro-F1 delta: {final_macro_f1 - V1_REFERENCE_MACRO_F1:+.4f}")

cm = confusion_matrix(test_p["y"], final_preds, labels=[0, 1])

print("\nConfusion matrix (label order: [0, 1] = [REAL, FAKE]):")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        test_p["y"],
        final_preds,
        labels=[0, 1],
        target_names=["REAL", "FAKE"],
        digits=4
    )
)

## 6) Optional: save a markdown run summary

This helper file can be copied into `results/liar_baseline.md`.

In [ ]:
def dataframe_to_markdown(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = []
    lines.append("| " + " | ".join(cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(cols)) + " |")

    for _, row in df.iterrows():
        values = [str(row[col]) for col in cols]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

run_date = datetime.now().strftime("%Y-%m-%d %H:%M")

valid_export = valid_results_df.copy()
valid_export["accuracy"] = valid_export["accuracy"].map(lambda x: f"{x:.4f}")
valid_export["macro_f1"] = valid_export["macro_f1"].map(lambda x: f"{x:.4f}")

report_text = classification_report(
    test_p["y"],
    final_preds,
    labels=[0, 1],
    target_names=["REAL", "FAKE"],
    digits=4
)

summary_lines = []
summary_lines.append("# LIAR Baseline V2 Run Output")
summary_lines.append("")
summary_lines.append(f"- Date: {run_date}")
summary_lines.append("- Task: Binary classification")
summary_lines.append("- Label mapping: REAL=0, FAKE=1")
summary_lines.append(f"- Version 1 Accuracy: {V1_REFERENCE_ACC:.4f}")
summary_lines.append(f"- Version 1 Macro-F1: {V1_REFERENCE_MACRO_F1:.4f}")
summary_lines.append("")
summary_lines.append("## Selected Version 2 configuration")
summary_lines.append(f"- Preprocess: {selected_preprocess_name}")
summary_lines.append(f"- ngram_range: {selected_ngram_range}")
summary_lines.append(f"- min_df: {selected_min_df}")
summary_lines.append(f"- max_df: {selected_max_df}")
summary_lines.append("")
summary_lines.append("## Final Version 2 test result")
summary_lines.append(f"- Accuracy: {final_acc:.4f}")
summary_lines.append(f"- Macro-F1: {final_macro_f1:.4f}")
summary_lines.append(f"- Accuracy delta vs V1: {final_acc - V1_REFERENCE_ACC:+.4f}")
summary_lines.append(f"- Macro-F1 delta vs V1: {final_macro_f1 - V1_REFERENCE_MACRO_F1:+.4f}")
summary_lines.append("")
summary_lines.append("## Validation experiments")
summary_lines.append("")
summary_lines.append(dataframe_to_markdown(valid_export))
summary_lines.append("")
summary_lines.append("## Confusion matrix")
summary_lines.append("- Label order: [0, 1] = [REAL, FAKE]")
summary_lines.append("```")
summary_lines.append(str(cm))
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Classification report")
summary_lines.append("```")
summary_lines.append(report_text)
summary_lines.append("```")

out_dir = Path("results")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "liar_baseline_v2_run_output.md"
out_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved:", out_path.resolve())

## What this notebook already addresses

This notebook directly addresses the supervisor's technical requests:
- inspect class balance
- improve preprocessing
- review TF-IDF parameters
- use the validation split properly
- keep the final test evaluation separate

A separate written file is still needed for:
- literature comparison with other LIAR results
- repo/progress summary updates